# Load Datasets and device related CVES

In [2]:
import pandas as pd

# Load the dataset
cve_dataset = pd.read_excel('Datasets/cve_data.xlsx')
cwe_dataset = pd.read_excel('Datasets/cwe_data.xlsx')
# List of infusion-related CVEs
infusion_cves = cve_dataset[['CVE ID']]


# Extract device related vulenrabilities

In [3]:
# Filter dataset to match only the CVEs in the infusion list
infusion_vulnerabilities = cve_dataset

# Display the filtered dataset
infusion_vulnerabilities.shape


(10, 14)

In [5]:
infusion_vulnerabilities

,CVE ID,Description,Weaknesses,baseScore,baseSeverity,attackVector,attackComplexity,privilegesRequired,userInteraction,confidentialityImpact,integrityImpact,availabilityImpact,exploitabilityScore,CVSS Version
0,CVE-2014-5433,An unauthenticated remote attacker may be able...,CWE-255; CWE-312,9.8,CRITICAL,NETWORK,LOW,NONE,NONE,HIGH,HIGH,HIGH,3.9,3.0
1,CVE-2020-12041,"The Baxter Spectrum WBM (v17, v20D29, v20D30, ...",CWE-732,9.4,CRITICAL,NETWORK,LOW,NONE,NONE,HIGH,LOW,HIGH,3.9,3.1
2,CVE-2014-5434,Baxter SIGMA Spectrum Infusion System version ...,CWE-259; CWE-798,9.8,CRITICAL,NETWORK,LOW,NONE,NONE,HIGH,HIGH,HIGH,3.9,3.0
3,CVE-2020-12047,"The Baxter Spectrum WBM (v17, v20D29, v20D30, ...",CWE-259; CWE-798,9.8,CRITICAL,NETWORK,LOW,NONE,NONE,HIGH,HIGH,HIGH,3.9,3.1
4,CVE-2020-12043,"The Baxter Spectrum WBM (v17, v20D29, v20D30, ...",CWE-672,9.8,CRITICAL,NETWORK,LOW,NONE,NONE,HIGH,HIGH,HIGH,3.9,3.1
5,CVE-2014-5431,Baxter SIGMA Spectrum Infusion System version ...,CWE-259; CWE-798,6.8,MEDIUM,PHYSICAL,LOW,NONE,NONE,HIGH,HIGH,HIGH,0.9,3.0
6,CVE-2020-12045,"The Baxter Spectrum WBM (v17, v20D29, v20D30, ...",CWE-259; CWE-798,9.8,CRITICAL,NETWORK,LOW,NONE,NONE,HIGH,HIGH,HIGH,3.9,3.1
7,CVE-2020-12040,Sigma Spectrum Infusion System v's6.x (model 3...,CWE-319,9.8,CRITICAL,NETWORK,LOW,NONE,NONE,HIGH,HIGH,HIGH,3.9,3.1
8,CVE-2020-12039,Baxter Sigma Spectrum Infusion Pumps Sigma Spe...,CWE-259; CWE-798,2.4,LOW,PHYSICAL,LOW,NONE,NONE,LOW,NONE,NONE,0.9,3.1
9,CVE-2014-5432,Baxter SIGMA Spectrum Infusion System version ...,CWE-287; CWE-592,9.8,CRITICAL,NETWORK,LOW,NONE,NONE,HIGH,HIGH,HIGH,3.9,3.0


# CVE Device vulnerability Preprocessing for Merge

In [67]:
infusion_vulnerabilities['Weaknesses'] = infusion_vulnerabilities['Weaknesses'].astype(str).fillna('')
# Split the 'Weaknesses' column into a list
infusion_vulnerabilities['Weaknesses'] = infusion_vulnerabilities['Weaknesses'].str.split('; ')

# Explode the 'Weaknesses' column so each CWE ID has its own row
infusion_vulnerabilities = infusion_vulnerabilities.explode('Weaknesses', ignore_index=True)

infusion_vulnerabilities_data = infusion_vulnerabilities
# Remove prefix 
infusion_vulnerabilities_data['Weaknesses'] = infusion_vulnerabilities_data['Weaknesses'].astype(str)
infusion_vulnerabilities_data['Weaknesses'] = infusion_vulnerabilities_data['Weaknesses'].str.replace("CWE-", "", regex=True)

# CWE DATA Preprocessing for Merge

In [68]:
# Rename CWE dataset columns with 'cwe_' prefix (only if not already prefixed)
if not any(col.startswith('cwe_') for col in cwe_dataset.columns):
    cwe_dataset = cwe_dataset.add_prefix('cwe_')

# Rename 'cwe_CWE_ID' column to match 'Weaknesses' column
cwe_dataset.rename(columns={'cwe_CWE_ID': 'Weaknesses'}, inplace=True)

# Rename 'cwe_CWE_ID' column to match 'Weaknesses' column
cwe_dataset.rename(columns={'cwe_CWE_ID': 'Weaknesses'}, inplace=True)

# Ensure 'Weaknesses' column in CWE dataset is also of type string
cwe_dataset['Weaknesses'] = cwe_dataset['Weaknesses'].astype(str)

# Merge CVE Device Vulnerabilities and CWE weaknesses

In [69]:
# Merge CWE details into infusion vulnerabilities dataset
device_weaknesses = infusion_vulnerabilities_data.merge(cwe_dataset, on='Weaknesses', how='left')

# Rename 'Weaknesses' column back to 'CWE ID' for clarity
device_weaknesses.rename(columns={'Weaknesses': 'CWE ID'}, inplace=True)

# Display the merged dataset
device_weaknesses.head()

,CVE ID,Description,CWE ID,baseScore,baseSeverity,attackVector,attackComplexity,privilegesRequired,userInteraction,confidentialityImpact,integrityImpact,availabilityImpact,exploitabilityScore,CVSS Version,cwe_Unnamed: 0,cwe_Name,cwe_Abstraction,cwe_Structure,cwe_Status,cwe_Description,cwe_Extended_Description,cwe_Related_Weaknesses,cwe_Weakness_Ordinality,cwe_Language_Name,cwe_Technology_Class,cwe_Alternate_Terms,cwe_Alternate_Terms_With_Descriptions,cwe_Background_Details,cwe_Modes_Of_Introduction,cwe_Likelihood_Of_Exploit,cwe_Common_Consequences,cwe_Detection_Methods,cwe_Potential_Mitigations,cwe_Demonstrative_Examples,cwe_Observed_Examples,cwe_Related_Attack_Patterns,cwe_Mapping_Notes
0,CVE-2014-5433,An unauthenticated remote attacker may be able...,255,9.8,CRITICAL,NETWORK,LOW,NONE,NONE,HIGH,HIGH,HIGH,3.9,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,CVE-2014-5433,An unauthenticated remote attacker may be able...,312,9.8,CRITICAL,NETWORK,LOW,NONE,NONE,HIGH,HIGH,HIGH,3.9,3.0,449.0,Cleartext Storage of Sensitive Information,Base,Simple,Draft,The product stores sensitive information in cl...,NaN,"CWE-311 (ChildOf, View: 1000, Ordinal: Primary...",NaN,NaN,"Mobile, Cloud Computing, ICS/OT",NaN,NaN,NaN,Architecture and Design: OMISSION: This weakne...,NaN,Scope: Confidentiality | Impact: Read Applicat...,Method: Automated Static Analysis | Effectiven...,Phase: Implementation | Description: N/A,Intro: The following code excerpt stores a pla...,Reference: CVE-2022-30275 | Description: Remot...,37,Usage: Allowed | Rationale: This CWE entry is ...
2,CVE-2020-12041,"The Baxter Spectrum WBM (v17, v20D29, v20D30, ...",732,9.4,CRITICAL,NETWORK,LOW,NONE,NONE,HIGH,LOW,HIGH,3.9,3.1,835.0,Incorrect Permission Assignment for Critical R...,Class,Simple,Draft,The product specifies permissions for a securi...,When a resource is given a permission setting ...,"CWE-285 (ChildOf, View: 1000, Ordinal: Primary...",NaN,NaN,"Cloud Computing, Not Technology-Specific",NaN,NaN,NaN,Architecture and Design: N/A | Implementation:...,High,Scope: Confidentiality | Impact: Read Applicat...,Method: Automated Static Analysis | Effectiven...,Phase: Implementation | Description: N/A || Ph...,Intro: The following code sets the umask of th...,Reference: CVE-2022-29527 | Description: Go ap...,"1, 122, 127, 17, 180, 206, 234, 60, 61, 62, 642",Usage: Allowed-with-Review | Rationale: While ...
3,CVE-2014-5434,Baxter SIGMA Spectrum Infusion System version ...,259,9.8,CRITICAL,NETWORK,LOW,NONE,NONE,HIGH,HIGH,HIGH,3.9,3.0,394.0,Use of Hard-coded Password,Variant,Simple,Draft,"The product contains a hard-coded password, wh...",NaN,"CWE-798 (ChildOf, View: 1000, Ordinal: Primary...",Primary,NaN,ICS/OT,NaN,NaN,NaN,Implementation: REALIZATION: This weakness is ...,High,Scope: Access Control | Impact: Gain Privilege...,Method: Manual Analysis | Effectiveness: N/A |...,Phase: Architecture and Design | Description: ...,Intro: The following code uses a hard-coded pa...,Reference: CVE-2022-29964 | Description: Distr...,NaN,Usage: Allowed | Rationale: This CWE entry is ...
4,CVE-2014-5434,Baxter SIGMA Spectrum Infusion System version ...,798,9.8,CRITICAL,NETWORK,LOW,NONE,NONE,HIGH,HIGH,HIGH,3.9,3.0,888.0,Use of Hard-coded Credentials,Base,Simple,Draft,"The product contains hard-coded credentials, s...",NaN,"CWE-1391 (ChildOf, View: 1000, Ordinal: Primar...",Primary,NaN,"Mobile, ICS/OT",NaN,NaN,NaN,Architecture and Design: REALIZATION: This wea...,High,Scope: Access Control | Impact: Bypass Protect...,Method: Black Box | Effectiveness: Moderate | ...,Phase: Architecture and Design | Description: ...,Intro: The following code uses a hard-coded pa...,Reference: CVE-2022-29953 | Description: Condi...,"191, 70",Usage: Allowed | Rationale: This CWE entry is ...


In [70]:
pd.set_option('display.max_columns', None)  # Show all columns

# Load CAPEC Data and Preprocess for Merge

In [71]:
# Load CAPEC dataset
capec_dataset = pd.read_excel('Datasets/capec_data.xlsx')


# Preprocess CAPEC Data for Merge

In [72]:
# Ensure CAPEC_ID is a string and strip whitespace
capec_dataset['CAPEC_ID'] = capec_dataset['CAPEC_ID'].astype(str).str.strip()

# Prefix CAPEC dataset columns with 'capec_' (only if not already prefixed)
capec_dataset.columns = [f'capec_{col}' if not col.startswith('capec_') else col for col in capec_dataset.columns]

# Rename CAPEC_ID column in capec_dataset to match CWE column for merging
capec_dataset.rename(columns={'capec_CAPEC_ID': 'cwe_Related_Attack_Patterns'}, inplace=True)



# Preprocess DeviceWeaknesses df for merge

In [73]:

# Ensure Related_Attack_Patterns in merged_df is string and clean it
device_weaknesses['cwe_Related_Attack_Patterns'] = device_weaknesses['cwe_Related_Attack_Patterns'].astype(str).str.strip()

# Split multiple CAPEC IDs into separate rows
device_weaknesses['cwe_Related_Attack_Patterns'] = device_weaknesses['cwe_Related_Attack_Patterns'].str.split(',')
device_weaknesses = device_weaknesses.explode('cwe_Related_Attack_Patterns')

# Clean whitespace again after explode
device_weaknesses['cwe_Related_Attack_Patterns'] = device_weaknesses['cwe_Related_Attack_Patterns'].str.strip()


# Merge Device weaknesses with Capec Attack Patterns

In [74]:
# Merge CAPEC details based on Related Attack Patterns
device_Attack_Pattern = device_weaknesses.merge(capec_dataset, on='cwe_Related_Attack_Patterns', how='left')

# Rename back to CAPEC_ID for clarity
device_Attack_Pattern.rename(columns={'cwe_Related_Attack_Patterns': 'CAPEC_ID'}, inplace=True)

# Debug: Check for missing CAPEC names
missing = device_Attack_Pattern[device_Attack_Pattern['capec_Name'].isnull()]
print("Missing CAPEC Names for these IDs:\n", missing['CAPEC_ID'].unique())
device_Attack_Pattern = device_Attack_Pattern.loc[:, ~device_Attack_Pattern.columns.str.contains('^Unnamed')]

# Show final DataFrame
device_Attack_Pattern.head()


Missing CAPEC Names for these IDs:
 ['nan']


,CVE ID,Description,CWE ID,baseScore,baseSeverity,attackVector,attackComplexity,privilegesRequired,userInteraction,confidentialityImpact,integrityImpact,availabilityImpact,exploitabilityScore,CVSS Version,cwe_Unnamed: 0,cwe_Name,cwe_Abstraction,cwe_Structure,cwe_Status,cwe_Description,cwe_Extended_Description,cwe_Related_Weaknesses,cwe_Weakness_Ordinality,cwe_Language_Name,cwe_Technology_Class,cwe_Alternate_Terms,cwe_Alternate_Terms_With_Descriptions,cwe_Background_Details,cwe_Modes_Of_Introduction,cwe_Likelihood_Of_Exploit,cwe_Common_Consequences,cwe_Detection_Methods,cwe_Potential_Mitigations,cwe_Demonstrative_Examples,cwe_Observed_Examples,CAPEC_ID,cwe_Mapping_Notes,capec_Unnamed: 0,capec_Name,capec_Abstraction,capec_Status,capec_Description,capec_Extended_Description,capec_Likelihood_Of_Attack,capec_Typical_Severity,capec_Related_Attack_Patterns,capec_Execution_Flow,capec_Prerequisites,capec_Skills_Required,capec_Resources_Required,capec_Consequences,capec_Mitigations,capec_Example_Instances,capec_Related_Weaknesses,capec_Taxonomy_Mappings,capec_Mitre_Techniques
0,CVE-2014-5433,An unauthenticated remote attacker may be able...,255,9.8,CRITICAL,NETWORK,LOW,NONE,NONE,HIGH,HIGH,HIGH,3.9,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nan,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,CVE-2014-5433,An unauthenticated remote attacker may be able...,312,9.8,CRITICAL,NETWORK,LOW,NONE,NONE,HIGH,HIGH,HIGH,3.9,3.0,449.0,Cleartext Storage of Sensitive Information,Base,Simple,Draft,The product stores sensitive information in cl...,NaN,"CWE-311 (ChildOf, View: 1000, Ordinal: Primary...",NaN,NaN,"Mobile, Cloud Computing, ICS/OT",NaN,NaN,NaN,Architecture and Design: OMISSION: This weakne...,NaN,Scope: Confidentiality | Impact: Read Applicat...,Method: Automated Static Analysis | Effectiven...,Phase: Implementation | Description: N/A,Intro: The following code excerpt stores a pla...,Reference: CVE-2022-30275 | Description: Remot...,37,Usage: Allowed | Rationale: This CWE entry is ...,247.0,Retrieve Embedded Sensitive Data,Detailed,Draft,An attacker examines a target system to find s...,NaN,High,Very High,CAPEC-167 (ChildOf),Step 1: Explore - [Identify Target] Attacker i...,In order to feasibly execute this type of atta...,Medium: The attacker must possess knowledge of...,The attacker must possess access to the system...,Scope: Confidentiality | Impact: Read Data || ...,NaN,Using a tool such as 'strings' or similar to p...,"CWE-226, CWE-311, CWE-525, CWE-312, CWE-314, C...",ATTACK: 1005 - Data from Local System || ATTAC...,1005 - Data from Local System || 1552.004 - Un...
2,CVE-2020-12041,"The Baxter Spectrum WBM (v17, v20D29, v20D30, ...",732,9.4,CRITICAL,NETWORK,LOW,NONE,NONE,HIGH,LOW,HIGH,3.9,3.1,835.0,Incorrect Permission Assignment for Critical R...,Class,Simple,Draft,The product specifies permissions for a securi...,When a resource is given a permission setting ...,"CWE-285 (ChildOf, View: 1000, Ordinal: Primary...",NaN,NaN,"Cloud Computing, Not Technology-Specific",NaN,NaN,NaN,Architecture and Design: N/A | Implementation:...,High,Scope: Confidentiality | Impact: Read Applicat...,Method: Automated Static Analysis | Effectiven...,Phase: Implementation | Description: N/A || Ph...,Intro: The following code sets the umask of th...,Reference: CVE-2022-29527 | Description: Go ap...,1,Usage: Allowed-with-Review | Rationale: While ...,0.0,Accessing Functionality Not Properly Constrain...,Standard,Draft,"In applications, particularly web applications...",NaN,High,High,"CAPEC-122 (ChildOf), CAPEC-17 (CanPrecede)",Step 1: Explore - [Survey] The attacker survey...,The application must be navigable in a manner ...,Low: In order to discover unrestricted resourc...,None: No specialized resources are required to...,"Scope: Confidentiality, Access Control, Author...","In a J2EE setting, administrators can associat...",Implementing the Model-View-Controller (MVC) w...,"CWE-276

# Load Mitre ATT&CK DATASET

In [75]:
mitre_df = pd.read_excel('Datasets/techniques.xlsx')
mitre_df['supports remote'] = (
    mitre_df['supports remote']
    .fillna(0)      # replace NaN with 0
    .astype(int)    # now safe to convert
)

mitre_df['supports remote'] = mitre_df['supports remote'].astype('int')
# or if you prefer True/False:
mitre_df['supports remote'] = mitre_df['supports remote'].astype('bool')

mitre_df['impact type'] = mitre_df['impact type'].astype('category')

mitre_df = mitre_df.rename(columns={'description': 'tech_description'})



mitre_df.head()

,ID,STIX ID,name,tech_description,url,created,last modified,domain,version,tactics,detection,platforms,data sources,is sub-technique,sub-technique of,defenses bypassed,contributors,permissions required,supports remote,system requirements,impact type,effective permissions,relationship citations
0,T1548,attack-pattern--67720091-eee3-4d2d-ae16-826456...,Abuse Elevation Control Mechanism,Adversaries may circumvent mechanisms designed...,https://attack.mitre.org/techniques/T1548,30 January 2020,15 October 2024,enterprise-attack,1.4,"Defense Evasion, Privilege Escalation",Monitor the file system for files that have th...,"IaaS, Identity Provider, Linux, Office Suite, ...","Command: Command Execution, File: File Metadat...",False,NaN,NaN,NaN,"Administrator, User",False,NaN,NaN,NaN,"(Citation: TrendMicro RaspberryRobin 2022),(Ci..."
1,T1548.002,attack-pattern--120d5519-3098-4e1c-9191-2aa612...,Abuse Elevation Control Mechanism: Bypass User...,Adversaries may bypass UAC mechanisms to eleva...,https://attack.mitre.org/techniques/T1548/002,30 January 2020,21 April 2023,enterprise-attack,2.1,"Defense Evasion, Privilege Escalation",There are many ways to perform UAC bypasses wh...,Windows,"Command: Command Execution, Process: Process C...",True,T1548,Windows User Account Control,Casey Smith; Stefan Kanthak,"Administrator, User",False,NaN,NaN,Administrator,"(Citation: Microsoft BlackCat Jun 2022),(Citat..."
2,T1548.004,attack-pattern--b84903f0-c7d5-435d-a69e-de47cc...,Abuse Elevation Control Mechanism: Elevated Ex...,Adversaries may leverage the <code>Authorizati...,https://attack.mitre.org/techniques/T1548/004,30 January 2020,19 October 2022,enterprise-attack,1.0,"Defense Evasion, Privilege Escalation",Consider monitoring for <code>/usr/libexec/sec...,macOS,"Process: OS API Execution, Process: Process Cr...",True,T1548,NaN,"Erika Noerenberg, @gutterchurl, Carbon Black; ...","Administrator, User",False,NaN,NaN,root,"(Citation: Carbon Black Shlayer Feb 2019),"
3,T1548.001,attack-pattern--6831414d-bb70-42b7-8030-d4e06b...,Abuse Elevation Control Mechanism: Setuid and ...,An adversary may abuse configurations where an...,https://attack.mitre.org/techniques/T1548/001,30 January 2020,15 March 2023,enterprise-attack,1.1,"Defense Evasion, Privilege Escalation",Monitor the file system for files that have th...,"Linux, macOS","Command: Command Execution, File: File Metadat...",True,T1548,NaN,NaN,User,False,NaN,NaN,NaN,"(Citation: OSX Keydnap malware),(Citation: ANS..."
4,T1548.003,attack-pattern--1365fe3b-0f50-455d-b4da-266ce3...,Abuse Elevation Control Mechanism: Sudo and Su...,Adversaries may perform sudo caching and/or us...,https://attack.mitre.org/techniques/T1548/003,30 January 2020,14 March 2022,enterprise-attack,1.0,"Defense Evasion, Privilege Escalation","On Linux, auditd can alert every time a user's...","Linux, macOS","Command: Command Execution, File: File Modific...",True,T1548,NaN,NaN,User,False,NaN,NaN,root,"(Citation: hexed osx.dok analysis 2019),(Citat..."


# Preprocess and Merge Device_Attack_Patterns with Mitre Techniques And Tactics dataset

In [76]:


# Step 0: Split the column by '||' and explode it
device_Attack_Pattern['capec_Mitre_Techniques'] = device_Attack_Pattern['capec_Mitre_Techniques'].str.split(r'\s*\|\|\s*')
device_Attack_Pattern = device_Attack_Pattern.explode('capec_Mitre_Techniques')

# Step 1: Extract technique ID from exploded entries (everything before ' -')
device_Attack_Pattern['Technique_ID_Only'] = device_Attack_Pattern['capec_Mitre_Techniques'].str.extract(r'^([\d\.]+)')

# Step 2: Remove the 'T' prefix in mitre_df's ID column
mitre_df['Technique_ID_Only'] = mitre_df['ID'].str.replace('T', '', regex=False)

# # Step 3: Merge on the numeric technique ID
device_Attack_Pattern = device_Attack_Pattern.merge(
    mitre_df,
    how='left',
    on='Technique_ID_Only'
)

# Optional: Display first few rows
print(device_Attack_Pattern.head())


           CVE ID                                        Description CWE ID  \
0   CVE-2014-5433  An unauthenticated remote attacker may be able...    255   
1   CVE-2014-5433  An unauthenticated remote attacker may be able...    312   
2   CVE-2014-5433  An unauthenticated remote attacker may be able...    312   
3  CVE-2020-12041  The Baxter Spectrum WBM (v17, v20D29, v20D30, ...    732   
4  CVE-2020-12041  The Baxter Spectrum WBM (v17, v20D29, v20D30, ...    732   

   baseScore baseSeverity attackVector attackComplexity privilegesRequired  \
0        9.8     CRITICAL      NETWORK              LOW               NONE   
1        9.8     CRITICAL      NETWORK              LOW               NONE   
2        9.8     CRITICAL      NETWORK              LOW               NONE   
3        9.4     CRITICAL      NETWORK              LOW               NONE   
4        9.4     CRITICAL      NETWORK              LOW               NONE   

  userInteraction confidentialityImpact integrityImpact 

In [77]:
# Initialize new columns
device_Attack_Pattern['technique'] = None
device_Attack_Pattern['subtechnique'] = None

def classify_id(row):
    tech_id = row['Technique_ID_Only']
    if pd.isna(tech_id):
        return pd.Series([None, None])
    if '.' in str(tech_id):
        parent_id = str(tech_id).split('.')[0]
        return pd.Series([parent_id, tech_id])
    else:
        return pd.Series([tech_id, None])


# Apply classification
device_Attack_Pattern[['technique', 'subtechnique']] = device_Attack_Pattern.apply(classify_id, axis=1)


In [78]:
# Create a cleaned version of Technique_ID_Only in mitre_df
mitre_df['Technique_ID_Only'] = mitre_df['ID'].str.replace('T', '', regex=False)

# Technique name and tactics lookup
id_to_name = mitre_df.set_index('Technique_ID_Only')['name'].to_dict()
id_to_tactics = mitre_df.set_index('Technique_ID_Only')['tactics'].to_dict()

# For technique column
device_Attack_Pattern['technique_name'] = device_Attack_Pattern['technique'].map(id_to_name)
device_Attack_Pattern['tactics'] = device_Attack_Pattern['technique'].map(id_to_tactics)


device_Attack_Pattern['subtechnique_name'] = device_Attack_Pattern['subtechnique'].map(id_to_name)


In [79]:
# Define the list of columns you want to keep
# Create the final dataframe with selected columns
final_df = device_Attack_Pattern[[
    'CVE ID',
    'Description',
    'attackVector',
    'baseScore',
    'baseSeverity',
    'attackComplexity',
    'privilegesRequired',
    'userInteraction',
    'confidentialityImpact',
    'availabilityImpact',
    'integrityImpact',
    'exploitabilityScore',
    
    #CAPEC details
    'capec_Name',
    'capec_Mitre_Techniques',        # assuming this is what you mean by 'tactics'
    "capec_Abstraction",
    "capec_Status",
    "capec_Description",
    "capec_Extended_Description",
    "capec_Likelihood_Of_Attack",
    "capec_Typical_Severity",
    "capec_Execution_Flow",
    "capec_Prerequisites",
    "capec_Skills_Required",
    'capec_Resources_Required',
    "capec_Consequences",
    "capec_Mitigations",
    "capec_Example_Instances",
    "capec_Related_Weaknesses",
    "capec_Taxonomy_Mappings",

    # Additional CWE details requested
    'cwe_Name',
    'cwe_Abstraction',
    'cwe_Structure',
    'cwe_Status',
    'cwe_Description',
    'cwe_Extended_Description',
    'cwe_Related_Weaknesses',
    'cwe_Weakness_Ordinality',
    'cwe_Language_Name',
    'cwe_Technology_Class',
    'cwe_Alternate_Terms',
    'cwe_Alternate_Terms_With_Descriptions',
    'cwe_Background_Details',
    'cwe_Modes_Of_Introduction',
    'cwe_Likelihood_Of_Exploit',
    'cwe_Common_Consequences',
    'cwe_Detection_Methods',
    'cwe_Potential_Mitigations',
    'cwe_Demonstrative_Examples',
    'cwe_Observed_Examples',

    # Additional MITRE ATT&CK technique/sub-technique fields
    'technique_name',
    'tech_description',
    'subtechnique_name',
    'tactics',
    'detection',
    'platforms',
    'data sources',
    'defenses bypassed',
    'contributors',
    'permissions required',
    'supports remote',
    'system requirements',
    'impact type',
    'effective permissions',

]]
# Step 1: Ensure the 'tactics' column is a list
final_df['tactics'] = final_df['tactics'].str.split(r'\s*,\s*')

# Step 2: Explode the 'tactics' column
final_df = final_df.explode('tactics')

final_df


/tmp/ipykernel_75727/2659015850.py:76: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_df['tactics'] = final_df['tactics'].str.split(r'\s*,\s*')


,CVE ID,Description,attackVector,baseScore,baseSeverity,attackComplexity,privilegesRequired,userInteraction,confidentialityImpact,availabilityImpact,integrityImpact,exploitabilityScore,capec_Name,capec_Mitre_Techniques,capec_Abstraction,capec_Status,capec_Description,capec_Extended_Description,capec_Likelihood_Of_Attack,capec_Typical_Severity,capec_Execution_Flow,capec_Prerequisites,capec_Skills_Required,capec_Resources_Required,capec_Consequences,capec_Mitigations,capec_Example_Instances,capec_Related_Weaknesses,capec_Taxonomy_Mappings,cwe_Name,cwe_Abstraction,cwe_Structure,cwe_Status,cwe_Description,cwe_Extended_Description,cwe_Related_Weaknesses,cwe_Weakness_Ordinality,cwe_Language_Name,cwe_Technology_Class,cwe_Alternate_Terms,cwe_Alternate_Terms_With_Descriptions,cwe_Background_Details,cwe_Modes_Of_Introduction,cwe_Likelihood_Of_Exploit,cwe_Common_Consequences,cwe_Detection_Methods,cwe_Potential_Mitigations,cwe_Demonstrative_Examples,cwe_Observed_Examples,technique_name,tech_description,subtechnique_name,tactics,detection,platforms,data sources,defenses bypassed,contributors,permissions required,supports remote,system requirements,impact type,effective permissions
0,CVE-2014-5433,An unauthenticated remote attacker may be able...,NETWORK,9.8,CRITICAL,LOW,NONE,NONE,HIGH,HIGH,HIGH,3.9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,CVE-2014-5433,An unauthenticated remote attacker may be able...,NETWORK,9.8,CRITICAL,LOW,NONE,NONE,HIGH,HIGH,HIGH,3.9,Retrieve Embedded Sensitive Data,1005 - Data from Local System,Detailed,Draft,An attacker examines a target system to find s...,NaN,High,Very High,Step 1: Explore - [Identify Target] Attacker i...,In order to feasibly execute this type of atta...,Medium: The attacker must possess knowledge of...,The attacker must possess access to the system...,Scope: Confidentiality | Impact: Read Data || ...,NaN,Using a tool such as 'strings' or similar to p...,"CWE-226, CWE-311, CWE-525, CWE-312, CWE-314, C...",ATTACK: 1005 - Data from Local System || ATTAC...,Cleartext Storage of Sensitive Information,Base,Simple,Draft,The product stores sensitive information in cl...,NaN,"CWE-311 (ChildOf, View: 1000, Ordinal: Primary...",NaN,NaN,"Mobile, Cloud Computing, ICS/OT",NaN,NaN,NaN,Architecture and Design: OMISSION: This weakne...,NaN,Scope: Confidentiality | Impact: Read Applicat...,Method: Automated Static Analysis | Effectiven...,Phase: Implementation | Description: N/A,Intro: The following code excerpt stores a pla...,Reference: CVE-2022-30275 | Description: Remot...,Data from Local System,"Adversaries may search local system sources, s...",NaN,Collection,Monitor processes and command-line arguments f...,"Linux, Network, Windows, macOS","Command: Command Execution, File: File Access,...",NaN,"Austin Clark, @c2defense; William Cain",NaN,False,Privileges to access certain files and directo...,NaN,NaN
2,CVE-2014-5433,An unauthenticated remote attacker may be able...,NETWORK,9.8,CRITICAL,LOW,NONE,NONE,HIGH,HIGH,HIGH,3.9,Retrieve Embedded Sensitive Data,1552.004 - Unsecured Credentials,Detailed,Draft,An attacker examines a target system to find s...,NaN,High,Very High,Step 1: Explore - [Identify Target] Attacker i...,In order to feasibly execute this type of atta...,Medium: The attacker must possess knowledge of...,The attacker must possess access to the system...,Scope: Confidentiality | Impact: Read Data || ...,NaN,Using a tool such as 'strings' or similar to p...,"CWE-226, CWE-311, CWE-525, CWE-312, CWE-314, C...",ATTACK: 1005 - Data from Local System || ATTAC...,Cleartext Storage of Sensitive Information,Base,Simple,Draft,The product stores sensitive information in cl...,NaN,"CWE-311 (ChildOf, View: 1000, Ordinal: Primary...",NaN,NaN,"Mobile, Cloud Computing, ICS/OT",NaN,NaN,NaN,Architecture and Design: OMISSION: This weakne...,NaN,Scope: Conf

In [80]:
device_Attack_Pattern.to_excel('Device_Attack_Pattern.xlsx', index=False)

In [81]:
final_df.to_excel("../3.Visualizations/device_compromise_dataset.xlsx", index=False)